# 2-Dimensional Empirical Orthogonal Functions Statistical Analysis

In order to check the differences between the two differing ensembles — RSQSim and FakeQuakes —, it becomes necessary to investigate the statistics of the two. That is, do the two synthetic earthquake generation models, one with physics and one with statistics, have the same physical properties?

The null hypothesis $H_{o}$ is that the two model ensembles have the same physics. The other hypothesis $H_{1}$ is that they do not. In order to test these hypotheses, I utilize a 2D EOF analysis based off [Yang et al. (2004)](https://dl.acm.org/doi/abs/10.1109/TPAMI.2004.1261097). The math is explored in the following cells.

In [1]:
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
geoclaw_cube=np.load("/Users/seansantellanes/Downloads/geoclaw_toy_database.npz")
geoclaw_data=geoclaw_cube["eta"][:-1]
hysea_cube=np.load("/Users/seansantellanes/Downloads/combined_eta_thousand.npz")
hysea_data=hysea_cube["eta"][:]
hysea_data=hysea_data.reshape(980,480,356)
hysea_data=np.transpose(hysea_data,(0,2,1))
print(geoclaw_data.shape)
print(hysea_data.shape)

(1000, 356, 480)
(980, 356, 480)


# Initial Investigations

The GeoClaw data cube is of size (1000 x 356 x 480), where 1000 is the number of realizations from FakeQuakes, 356 the number of stations, and 480 the time sampling (here 480 sample for sampling periods of 15 s for 2 hours of simulation time).

The HySEA data cube is of (980 x 356 x 480). The numbers mean the same as before. However, we are kept to 980 due to the numeric sampling of RSQsim catalog. 

In [2]:
from scipy.linalg import svd
from scipy.spatial.distance import cdist

def energy_distance(X, Y):
    """
    X,Y : (N, stations, time)

    Returns
    -------
    scalar product
    """
    dxy = cdist(X, Y).mean()
    dxx = cdist(X, X).mean()
    dyy = cdist(Y, Y).mean()

    return 2*dxy - dxx - dyy
def compute_2d_eofs(X, n_space=20, n_time=20):
    """
    X : (N, stations, time)

    Returns
    -------
    U : spatial EOFs
    V : temporal EOFs
    mean_field
    """
    mean_field = X.mean(axis=0)
    Xc = X - mean_field

    # Spatial covariance
    Cspace = np.zeros((X.shape[1], X.shape[1]))

    # Temporal covariance
    Ctime = np.zeros((X.shape[2], X.shape[2]))

    for sample in Xc:
        Cspace += sample @ sample.T
        Ctime += sample.T @ sample

    Cspace /= X.shape[0]
    Ctime /= X.shape[0]

    U, _, _ = svd(Cspace)
    V, _, _ = svd(Ctime)

    return (
        U[:, :n_space],
        V[:, :n_time],
        mean_field,
    )
def project_2d(X, U, V, mean_field):

    """
    X : (N, stations, time)
    U : (stations, stations)
    V : (time, time)

    Returns
    -------
    coeffs : arraylike
     
    """

    Xc = X - mean_field

    coeffs = []

    for sample in Xc:

        # coefficient matrix
        C = U.T @ sample @ V

        coeffs.append(C.ravel())

    return np.asarray(coeffs)
def eof2d_energy_test(
        A,
        B,
        n_space=20,
        n_time=20,
        n_permutations=1000,
        random_state=None):

    """
    A : (N, stations, time)
    B : (N, stations, time)
    n_space : number of spatial PCA modes
    n_time : number of temporal PCA modes
    n_permutations : how many times do you want to run the permutations.
    random_state : Initialize at the same random state or not! I'm not your boss.

    Returns
    -------
    
    "energy_distance": observed, what you want \n
    "p_value": p, what you need \n
    "U": U, can ignore \n
    "V": V, can ignore \n
    "null_distribution": perm can ignore \n

    """

    rng = np.random.default_rng(random_state)

    X = np.concatenate((A, B), axis=0)

    U, V, mean_field = compute_2d_eofs(
        X,
        n_space=n_space,
        n_time=n_time,
    )

    ZA = project_2d(A, U, V, mean_field)
    ZB = project_2d(B, U, V, mean_field)

    observed = energy_distance(ZA, ZB)

    Z = np.vstack((ZA, ZB))
    nA = len(ZA)
    nB = len(ZB)

    perm = np.empty(n_permutations)

    idx = np.arange(nA+nB)

    for i in range(n_permutations):

        rng.shuffle(idx)

        perm[i] = energy_distance(
            Z[idx[:nA]],
            Z[idx[nA:]]
        )

    p = (np.sum(perm >= observed)+1)/(n_permutations+1)

    return {
        "energy_distance": observed,
        "p_value": p,
        "U": U,
        "V": V,
        "null_distribution": perm,
    }

# Maths
Our problem is defined as the following:
$\begin{equation}
\text{ensemble} \times \text{station} \times \text{time} = (1000, 356, 480)
\end{equation}$

The 2D EOF does the following. Here $i$ means the ith relization of the ensembles.

$\begin{equation}
X_{i} \approx UC_{i}V^{T}
\end{equation}
$

* U contains the dominant spatial EOFs (356 x $r_{s}$)
* $C_{i}$ is a small correlation coefficient matrix ($r_{s}$ by $r_{t}$)
* V contains the dominant temporal EOFs (480 x $r_{t}$)

The energy test is then performed over the flattened $C_{i}$ matrices. 

# The energy test

Consider the null hypothesis that two random variables, X and Y, have the same probability distributions $\mu = \nu$. For statistical samples from X and Y:

$x_{1},..., x_{n}$ and $y_{1},..., y_{m}$

the following arithmetic averages of distances are computed between the X and the Y samples: 

$\begin{equation}
A:=\frac{1}{nm}\sum_{i=1}^{n}\sum_{j=1}^{m}||x_{i}-y_{j}||,B:=\frac{1}{n^{2}}\sum_{i=1}^{n}\sum_{j=1}^{n}||x_{i}-x_{j}||,C:=\frac{1}{m^{2}}\sum_{i=1}^{m}\sum_{j=1}{m^{2}}||y_{i}-y_{j}||
\end{equation}
$

The E-statistic of the underlying null hypothesis is defined as follows: 
$\begin{equation}
E_{n,m}(X,Y):=2A-B-C
\end{equation}
$

One can prove that ${\displaystyle E_{n,m}(X,Y)\geq 0}$ and that the corresponding population value is zero if and only if X and Y have the same distribution (${\displaystyle \mu =\nu }$).

In [7]:
result = eof2d_energy_test(
    geoclaw_data,
    hysea_data,
    n_space=25,
    n_time=30,
    n_permutations=500,
    random_state=42,
)

print(result["energy_distance"])
print(result["p_value"])

3.5227226933967657
0.001996007984031936


# Interpretations

The results of the 500 n_permutations test is that our energy distance $E$ is 3.52. This distance means that the two do not inhabit the same statistical distribution. To add strength to this argument, we also calculate the p-value, which for this test is 0.0020. As $p \leq 0.05$, that means that the ensembles differ statistically. 

Upon investigations of the two data cubes, it becomes clear that these tests are valid. 

<figure>
  <img src="data_cube_vis.png" alt="Comparison of FakeQuakes and RSQSim" width="700">
  <figcaption><b>Figure 1.</b> Comparison of FakeQuakes and RSQSim. Gold lines show FakeQuakes median percentile with the shaded region showing 5 to 95 percentile range. Blue
  lines show the RSQSim median percentile with the 5 to 95 percentile range shown by the shaded blue region.
  </figcaption>
</figure>

From the above image and stats test, it is clear that the two do not overlap statistically. Median $M_{w}$ values are 7.5 for FakeQuakes and 7.8 for RSQSim. The docket of things to do is to consider whether the statistical differences are due to FakeQuakes producing smaller tsunamis because its prescribed $M_{w}$ range is less than that of RSQSim.